# SAM Instance Debugger

Build the cache first with `build_rgb_cache.py`, then use this notebook to step the standalone SAM-instance algorithm frame by frame.

The notebook intentionally keeps the control surface small:
- adjust the hyperparameters in one cell
- recreate the debugger after edits
- use the widget to step / seek / jump to the next seed frame
- inspect detailed decisions and bucket state in the text panel

In [1]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '6'

In [2]:
from pathlib import Path

from map_runtime.sam_instance_debug import CachedSAMInstanceDebugger, DebuggerConfig, create_debugger_widget
from map_runtime.sam_masks import SAMAutomaticMaskConfig

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [3]:
# ---------------------------------------------------------------------------------->
# 'metrics': {'instance': {'ap_25': 0.2921115921115921,
#    'ap_50': 0.18088290310512534,
#    'ap_55': 0.18088290310512534,
#    'ap_60': 0.18088290310512534,
#    'ap_65': 0.09343434343434344,
#    'ap_70': 0.06565656565656566,
#    'ap_75': 0.06565656565656566,
#    'ap_80': 0.06565656565656566,
#    'ap_85': 0.05555555555555555,
#    'ap_90': 0.0,
#    'ap_95': 0.0,
#    'ap': 0.0888608305274972}},

# 'metrics': {'instance': {'ap_25': 0.5841750841750842,
#    'ap_50': 0.41750841750841755,
#    'ap_55': 0.25084175084175087,
#    'ap_60': 0.25084175084175087,
#    'ap_65': 0.25084175084175087,
#    'ap_70': 0.25084175084175087,
#    'ap_75': 0.25084175084175087,
#    'ap_80': 0.17676767676767677,
#    'ap_85': 0.03787878787878788,
#    'ap_90': 0.027777777777777776,
#    'ap_95': 0.0,
#    'ap': 0.19141414141414143}},
# ---------------------------------------------------------------------------------->

CACHE_DIR = Path("data/output/rgb_caches/ScanNet/scene0011_00")

sam_amg_config = SAMAutomaticMaskConfig(
    sort_mode="area",   # can be "score"
    min_mask_area_perc=0.01,
    points_per_side=24,   # can be lower like 16
    points_per_batch=128,
    pred_iou_thresh=0.88,
    stability_score_thresh=0.92,
    stability_score_offset=1.0,
    mask_threshold=0.0,
    box_nms_thresh=0.7,
    crop_n_layers=0,
    crop_nms_thresh=0.7,
    crop_overlap_ratio=0.6,
    crop_n_points_downscale_factor=1,
    point_grids=None,
    min_mask_region_area=0,
    output_mode="binary_mask",
    use_m2m=False,
    multimask_output=True,
    score_pred_iou_power=2.0,
    score_stability_power=1.0,
    score_area_power=0.0,   # can be 1.0
    mask_overlap_rescore_thresh=0.0,
    mask_overlap_rescore_power=1.0,
    mask_dedupe_iou_thresh=0.85,
    mask_containment_thresh=0.0,
)

config = DebuggerConfig(
    point_gid_slots=10,
    sam_model_level=24,   # can be 13
    sam_track_model_level=24,
    sam2_max_num_objects=16,
    sam_amg_config=sam_amg_config,
    reuse_inside_frac_th=0.40,
    reuse_outside_frac_th=0.10,
    min_mask_points=1,
    min_track_visible_points=1,
    prune_every_frames=64,  # 50
    prune_stale_gap_frames=100000,  # 2000
    prune_min_support_ratio=0.0,  # 0.0005
    prune_min_points=1000,  # 5000
)

DEVICE = "cuda"

In [4]:
debugger = CachedSAMInstanceDebugger(CACHE_DIR, config, device=DEVICE)
create_debugger_widget(debugger)

In [ ]:
debugger.show_sam_masks()

In [ ]:
debugger.show(gid=20)

In [ ]:
# Bucket / point-membership queries on the current debugger state.
# Run debugger.step(), debugger.seek(...), or use the widget first so debugger.current_view is populated.
#
# gid = 3
# bucket = debugger.buckets[gid]
# bucket
#
# point_id = 12345
# debugger.point_gids[point_id]  # K=10 gid slots for one global 3D point
#
# row, col = 200, 300
# point_id = int(debugger.current_view["point_ids_after"][row, col])
# if point_id >= 0:
#     print("pixel -> point_id", point_id)
#     print("gid slots", debugger.point_gids[point_id])
#
# active_buckets = {gid: debugger.buckets[gid] for gid in debugger.current_view["seeded_gids"]}
# active_buckets


In [ ]:
# Manual inspection examples
# debugger.step()
# debugger.seek(80)
# print(debugger.current_text_summary())
# debugger.current_view["decisions"]
# debugger.show(local_id=3)  # seed frames only
# debugger.show(gid=3)       # current projected global-instance support for gid=3
# debugger.show(local_id=3, gid=3)  # compare seed local mask vs current projected gid support
# debugger.simulate_sam_onlyseed_video(upto=100, map_every=1)
# debugger.simulate_video(upto=-1)

In [ ]:
debugger.simulate_sam_onlyseed_video(upto=200, map_every=1)

In [5]:
debugger.get_metrics(scannet_raw_root="/robodata/smodak/datasets/scannet_v2/scans", min_component_size=2000, ovo_score_th=0.0, chunk_size=100_000)

scene0011_00 debugger metrics:   0%|                                                                          …

ovo instance pooling:   0%|                                                                                   …

instance label video:   0%|                                                                                   …

{'frame_id': 2373,
 'dataset_name': 'ScanNet',
 'scene_name': 'scene0011_00',
 'metrics': {'instance': {'ap_25': 0.6216509327352825,
   'ap_50': 0.24261861043271296,
   'ap_55': 0.2116924701859938,
   'ap_60': 0.16372943309812338,
   'ap_65': 0.08741761907396584,
   'ap_70': 0.05358899786144368,
   'ap_75': 0.05358899786144368,
   'ap_80': 0.026920791626673983,
   'ap_85': 0.014077425842131725,
   'ap_90': 0.005896064719594131,
   'ap_95': 0.0005941770647653001,
   'ap': 0.08601245877668484}},
 'diagnostics': {'instance': {'gt_instance_count': 33,
   'pred_instance_count': 112,
   'ignored_gt_instance_count': 0,
   'instance_metric_mode': 'class_agnostic_ap',
   'instance_score_source': 'seed_support_ratio',
   'num_eval_classes': 1,
   'num_eval_entries': 1,
   'gt_instances_per_class': {0: 33},
   'pred_instances_per_class': {0: 112},
   'matched_instance_count': 112,
   'assigned_gt_vertices': 237360,
   'transfer_k': 5,
   'source_pred_points': 6009112},
  'semantic_ovo_style': {'n

In [6]:
debugger.get_metrics(scannet_raw_root="/robodata/smodak/datasets/scannet_v2/scans", min_component_size=2000, ovo_score_th=0.0, chunk_size=100_000, use_optimal_collapse=True)

scene0011_00 debugger metrics:   0%|                                                                          …

ovo instance pooling:   0%|                                                                                   …

instance label video:   0%|                                                                                   …

{'frame_id': 2373,
 'dataset_name': 'ScanNet',
 'scene_name': 'scene0011_00',
 'metrics': {'instance': {'ap_25': 0.9696969696969697,
   'ap_50': 0.6608864676513418,
   'ap_55': 0.5473418394471026,
   'ap_60': 0.4384370015948964,
   'ap_65': 0.39383778360858157,
   'ap_70': 0.3168938965203821,
   'ap_75': 0.2623565879508154,
   'ap_80': 0.12505587853635391,
   'ap_85': 0.0772005772005772,
   'ap_90': 0.02554112554112554,
   'ap_95': 0.007215007215007214,
   'ap': 0.2854766165266184}},
 'diagnostics': {'instance': {'gt_instance_count': 33,
   'pred_instance_count': 32,
   'ignored_gt_instance_count': 0,
   'instance_metric_mode': 'class_agnostic_ap',
   'instance_score_source': 'seed_support_ratio',
   'num_eval_classes': 1,
   'num_eval_entries': 1,
   'gt_instances_per_class': {0: 33},
   'pred_instances_per_class': {0: 32},
   'matched_instance_count': 32,
   'assigned_gt_vertices': 237360,
   'transfer_k': 5,
   'source_pred_points': 4196873},
  'semantic_ovo_style': {'num_instances'